> Measure how volatile a model's forecasts are across successive runs, and whether its revisions were worth making.

## Motivation

Accuracy is usually the headline metric for a forecasting model, but it is not the only thing that matters in production.

A multi-horizon system running on a schedule produces several overlapping forecasts for the same future target date, one from each **forecast creation date** (FCD), as more data becomes available. In cross-validation output, the FCD is the `cutoff` column. That sequence of updates is a **forecast revision**, and how consistent or erratic those revisions are is what we call **forecast volatility**.

Consider an electrical grid operator using load forecasts to plan power supply. A forecast that revises from 45 GW to 65 GW ahead of a heat wave is a *useful* revision: it tells operators to activate reserve plants. But forecasts that jump around erratically between FCDs, without new information justifying the change, undermine trust and complicate planning. The goal is not to eliminate revisions. It is to distinguish benign, informative revisions from excessive, erratic ones.

Accuracy metrics cannot make that distinction: metrics such as `MAE` and `sCRPS` score each forecast against the truth and average, which discards the relationship *between* forecasts of the same date. Two models can post identical accuracy while one is steady and the other thrashes.

This tutorial introduces two metrics that measure forecast volatility. Part 1 applies them to two hand-built forecasters that accuracy metrics cannot tell apart. Part 2 applies them to the `cross_validation` output of StatsForecast and NeuralForecast models.

## The metrics

Both metrics compare the two forecasts that land on the **same target date** from consecutive FCDs. Indexing by series $b$, FCD $t$ and horizon step $h$, those are:

- $\hat{\mathbf{y}}_{b,t,h+1}$, issued at FCD $t$, looking $h+1$ steps ahead
- $\hat{\mathbf{y}}_{b,t+1,h}$, issued one FCD later, looking $h$ steps ahead

The second is a revision of the first.

### Symmetric Forecast Percentage Change (sFPC)

sFPC measures the relative change in predicted quantiles across consecutive FCDs, giving a quantitative view of the forecast revision rate:

$$\mathrm{sFPC}_{q} = \frac{200}{B \times T \times H} \sum_{b,t,h} \frac{|\hat{Y}^{(q)}_{b,t+1,h}-\hat{Y}^{(q)}_{b,t,h+1}|}{|\hat{Y}^{(q)}_{b,t+1,h}| + |\hat{Y}^{(q)}_{b,t,h+1}|}$$

Inspired by sMAPE, the denominator is symmetric in the two forecasts. This keeps the metric well behaved when predicted values are small and avoids the division-by-zero problems common to traditional percentage-based metrics.

### Scaled Excess Volatility (sEV)

sFPC treats every revision as equally undesirable, even ones that clearly improve accuracy. sEV instead penalises only revisions that move a forecast *away* from the truth, or that overshoot it, rewarding accuracy-improving revisions while separating them from harmful volatility:

$$\mathrm{EV}(y,\;\hat{\mathbf{y}}_1,\; \hat{\mathbf{y}}_2) = \mathrm{QL}(\hat{\mathbf{y}}_2,\hat{\mathbf{y}}_1) - \big(\mathrm{QL}(y,\hat{\mathbf{y}}_1)-\mathrm{QL}(y,\hat{\mathbf{y}}_2)\big)$$

$$\mathrm{sEV} = \frac{\sum_{b,t,h}\mathrm{EV}\big(y_{b,t,h},\;\hat{\mathbf{y}}_{b,t,h+1},\;\hat{\mathbf{y}}_{b,t+1,h}\big)}{\sum_{b,t,h}|y_{b,t,h}|}$$

where $\mathrm{QL}$ is the quantile loss, averaged over the quantile levels of the forecast:

$$\mathrm{QL}_q(y, \hat{y}^{(q)}) = q(y-\hat{y}^{(q)})_+ + (1-q)(\hat{y}^{(q)}-y)_+$$

The first EV term is what the revision cost; the bracketed term is the accuracy it bought. Because $\mathrm{QL}$ obeys the triangle inequality the second can never exceed the first, so $\mathrm{EV} \geq 0$. It is zero for a revision that moves straight toward the truth without overshooting it, and positive for one that moves away from the truth or overshoots it.

Like the other losses in `utilsforecast.losses`, both return one row per series and cutoff. A row scores the revisions made at that cutoff, so the first cutoff is empty, and averaging a series' rows over cutoffs gives the value over the whole cross-validation.

Both metrics need **overlapping** windows. If the step between FCDs is greater than or equal to the horizon, each date is forecast only once and there is no revision to measure.

## Setup

In [ ]:
%%capture
pip install utilsforecast statsforecast neuralforecast matplotlib -U

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MQLoss
from neuralforecast.models import MLP, NHITS
from neuralforecast.utils import AirPassengersPanel
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, Naive

from utilsforecast.losses import (
    excess_volatility,
    forecast_percentage_change,
    mae,
    scaled_crps,
)

logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning_fabric").setLevel(logging.ERROR)

## 1. Two forecasters that accuracy cannot tell apart

To see why a volatility metric is needed at all, we build two forecasters by hand and give them the *same* accuracy.

The series is a week of hourly demand with a daily shape and a weekday/weekend effect. We forecast it from six FCDs spaced 24 hours apart, each with a 48 hour horizon, so every window overlaps the next by 24 hours.

In [ ]:
HORIZON = 48
N_HOURS = 168
FCDS = [0, 24, 48, 72, 96, 120]

QUANTILES = [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9]
BAND = 3.5
# half-width of each quantile relative to the median, as a multiple of BAND
SPREAD = {0.1: 1.0, 0.25: 0.55, 0.4: 0.25, 0.5: 0.0, 0.6: -0.25, 0.75: -0.55, 0.9: -1.0}

hours = np.arange(N_HOURS)
times = pd.date_range("2024-01-01", periods=N_HOURS, freq="h")


def daily_shape(h):
    t = (h % 24) / 24
    return (
        40
        + 18 * np.sin(np.pi * (t - 0.05))
        + 10 * np.exp(-((t - 0.83) ** 2) / 0.01)
        - 6 * np.exp(-((t - 0.15) ** 2) / 0.008)
    )


weekday_effect = np.array([1.0, 1.02, 1.03, 1.02, 1.04, 0.92, 0.88])[hours // 24]
truth = daily_shape(hours) * weekday_effect + np.random.default_rng(7).normal(0, 0.5, N_HOURS)

The two forecasters differ only in **where their error comes from**.

The *steady* forecaster is wrong in a way that persists: it draws one error signal for the whole week, so every FCD is wrong about a given date in the same direction. Its forecasts for a shared date barely move.

The *erratic* forecaster draws fresh, independent error at every FCD, and additionally flips the sign of a shift on the overlapping half of each window. Its forecasts for a shared date swing back and forth.

The metrics take the long format that `cross_validation` returns, so we store each forecaster as its own series, with one row per FCD and target date and one column per quantile.

In [ ]:
QUANTILE_COLS = [f"q{int(100 * q)}" for q in QUANTILES]


def to_cv_df(unique_id, medians):
    """Stack per-FCD medians into a cross-validation frame with one column per quantile."""
    windows = []
    for start, median in zip(FCDS, medians):
        window = pd.DataFrame(
            {
                "unique_id": unique_id,
                "ds": times[start : start + HORIZON],
                "cutoff": times[start] - pd.Timedelta(hours=1),
                "y": truth[start : start + HORIZON],
            }
        )
        for q, col in zip(QUANTILES, QUANTILE_COLS):
            window[col] = median - SPREAD[q] * BAND
        windows.append(window)
    return pd.concat(windows, ignore_index=True)


# steady: one persistent error signal shared by every FCD, plus a little jitter
rng = np.random.default_rng(13)
persistent_error = rng.normal(0, 2.65, N_HOURS)
steady_medians = [
    truth[s : s + HORIZON] + persistent_error[s : s + HORIZON] + rng.normal(0, 0.8, HORIZON)
    for s in FCDS
]

# erratic: fresh error each FCD, plus a sign-flipping shift on the overlapping half
rng = np.random.default_rng(77)
erratic_medians = []
for i, s in enumerate(FCDS):
    shift = np.zeros(HORIZON)
    shift[HORIZON // 2 :] = (-1) ** i * 3.0
    erratic_medians.append(truth[s : s + HORIZON] + rng.normal(0, 2.0, HORIZON) + shift)

cv_df = pd.concat(
    [to_cv_df("steady", steady_medians), to_cv_df("erratic", erratic_medians)],
    ignore_index=True,
)

Plotting the six forecast windows against the truth makes the difference obvious.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, sharey=True)
colors = plt.cm.viridis(np.linspace(0, 0.9, len(FCDS)))

for ax, medians, title in [
    (axes[0], steady_medians, "Steady forecaster"),
    (axes[1], erratic_medians, "Erratic forecaster"),
]:
    ax.plot(times, truth, color="black", linewidth=1.2, label="actual", zorder=5)
    for i, (start, median) in enumerate(zip(FCDS, medians)):
        window = times[start : start + HORIZON]
        ax.plot(window, median, color=colors[i], linewidth=1.6, alpha=0.9)
        ax.fill_between(window, median - BAND, median + BAND, color=colors[i], alpha=0.12, linewidth=0)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_ylabel("demand")

axes[0].legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

Now score them, averaging each series over its cutoffs:

In [ ]:
models = {"model": QUANTILE_COLS}
quantiles = np.array(QUANTILES)

scores = {
    "MAE": mae(cv_df, ["q50"]),
    "sCRPS": scaled_crps(cv_df, models, quantiles),
    "sEV": excess_volatility(cv_df, models, quantiles),
    "sFPC": forecast_percentage_change(cv_df, ["q50"]),
}
pd.DataFrame(
    {name: res.groupby("unique_id").mean(numeric_only=True).squeeze() for name, res in scores.items()}
).round(4)

The two are effectively tied on accuracy: any accuracy-based comparison would call this a wash. On volatility they are not. The erratic forecaster revises its predictions several times as much as the steady one, and its scaled excess volatility is roughly double: the revisions it makes are not buying it accuracy.

This is the case for tracking volatility alongside accuracy rather than instead of it. The two measure different things, and a model can be good at one and bad at the other. Lower is better on both: the goal is to cut volatility without giving up accuracy.

## 2. Volatility metrics on cross-validation output

In practice the forecasts come from `cross_validation`, whose output already has the long format the metrics take.

The one requirement is that **`step_size` must be smaller than the horizon**, so that windows overlap.

To score the same nine quantiles as the paper, $0.1, 0.2, \dots, 0.9$, we forecast the intervals for `level=[20, 40, 60, 80]` and use the point forecast as the median. `score` pairs each quantile with its column and averages every metric over series and cutoffs.

In [ ]:
LEVELS = [20, 40, 60, 80]
QUANTILES_9 = np.round(np.arange(0.1, 1.0, 0.1), 1)


def quantile_cols(model):
    """Columns for the quantiles 0.1, ..., 0.9, with the point forecast as the median."""
    lo = [f"{model}-lo-{level}" for level in reversed(LEVELS)]
    hi = [f"{model}-hi-{level}" for level in LEVELS]
    return lo + [model] + hi


def score(cv_df, models):
    scores = {}
    for model in models:
        quantiles = {model: quantile_cols(model)}
        scores[model] = {
            "MAE": mae(cv_df, [model])[model].mean(),
            "sCRPS": scaled_crps(cv_df, quantiles, QUANTILES_9)[model].mean(),
            "sEV": excess_volatility(cv_df, quantiles, QUANTILES_9)[model].mean(),
            "sFPC": forecast_percentage_change(cv_df, [model])[model].mean(),
        }
    return pd.DataFrame(scores).round(4)

### StatsForecast

The prediction intervals of `AutoETS` and `Naive` are symmetric around their point forecast, so it is also their median.

In [ ]:
df = AirPassengersPanel[["unique_id", "ds", "y"]]
sf = StatsForecast(models=[AutoETS(season_length=12), Naive()], freq="ME")
sf_cv_df = sf.cross_validation(df=df, h=12, n_windows=8, step_size=4, level=LEVELS)
score(sf_cv_df, ["AutoETS", "Naive"])

### NeuralForecast

With `MQLoss`, NeuralForecast names its median `<model>-median`, so we rename it to the model name.

In [ ]:
kwargs = dict(
    h=12,
    input_size=24,
    max_steps=100,
    loss=MQLoss(level=LEVELS),
    random_seed=0,
    enable_progress_bar=False,
    enable_model_summary=False,
    logger=False,
    devices=1,
)
nf = NeuralForecast(models=[NHITS(**kwargs), MLP(**kwargs)], freq="ME")
nf_cv_df = nf.cross_validation(df=df, n_windows=8, step_size=4)
nf_cv_df = nf_cv_df.rename(columns={"NHITS-median": "NHITS", "MLP-median": "MLP"})
score(nf_cv_df, ["NHITS", "MLP"])

Read these together rather than separately. Accuracy tells you how close the forecasts land; `sFPC` tells you how much the model moves them between FCDs; `sEV` tells you how much of that movement was wasted. Lower is better for both, and driving them down is what reducing forecast volatility means in practice.

A few practical notes:

- `excess_volatility` takes any set of quantiles, so it also works on a point forecast alone: `models={"AutoETS": ["AutoETS"]}` with `quantiles=[0.5]` scores revisions with the absolute error.
- Both metrics also work inside `evaluate`, e.g. `evaluate(cv_df, metrics=[mae, excess_volatility], level=[80])`. There `excess_volatility` uses only the interval quantiles from `level` and leaves out the median, since a model's point forecast is not always its median.
- `forecast_percentage_change` is a point metric: pass a point forecast, or the median of a probabilistic one, as in the paper.
- sEV scales every cutoff of a series by the same factor, $\sum|y|$ over all its pairs divided by the number of cutoffs, which is what lets the rows average to the whole cross-validation. Adding or removing a cutoff therefore changes the sEV of the other cutoffs.

## References

- Willa Potosnak, Malcolm Wolff, Mengfei Cao, Ruijun Ma, Tatiana Konstantinova, Dmitry Efimov, Michael W. Mahoney, Boris Oreshkin, Kin G. Olivares. [Forking-Sequences: Statistically and Computationally Efficient Multi-Horizon Forecasting with Reduced Volatility](https://openreview.net/forum?id=dXdycy7WCX). Transactions on Machine Learning Research (2026).